# Task 2b: Local Window Sum With `torch.nn.RNN`

For each position `t`, the target is the sum of the input values inside a centered window of radius `K`.

## TODO

Implement `RNNModel` using only `torch.nn` modules.

- Use `nn.Embedding` for integer tokens.
- Use `nn.RNN(..., batch_first=True)` for sequence processing.
- Use `nn.Linear` to classify each time step into a possible window sum.
- The largest possible target is `9 * (2 * K + 1)`, so `output_vocab` must be `9 * (2 * K + 1) + 1`.
- Return logits shaped `(B, T, output_vocab)` with no `softmax`.

In [14]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = 'cuda' if torch.cuda.is_available() else 'cpu'
set_seed(217)
K = 2
device

'cpu'

In [15]:
class CustomDataset(Dataset):
    def __init__(self, vocab_size=10, seq_len=25, size=40000, K=2):
        self.vocab_size = vocab_size
        self.seq_len = seq_len
        self.size = size
        self.K = K
        self.X = torch.randint(0, vocab_size, (size, seq_len))
        x_floated = self.X.unsqueeze(1).float()
        kernel = torch.ones((1, 1, 2 * K + 1))
        self.Y = F.conv1d(x_floated, kernel, padding='same').squeeze(1).long()

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

## Model TODO

The model needs context from nearby positions. A vanilla `nn.RNN` reads left-to-right, so start with a regular RNN and consider `bidirectional=True` if your accuracy is weak. If you make it bidirectional, remember the linear layer input size becomes `2 * d_model`.

In [19]:
class RNNModel(nn.Module):
    def __init__(self, K, d_model=64, num_layers=2):
        super().__init__()
        self.input_vocab = 10
        self.output_vocab = 9 * (2 * K + 1) + 1

        # TODO: create nn.Embedding(self.input_vocab, d_model)
        self.embedding=nn.Embedding(self.input_vocab,d_model)
        # TODO: create nn.RNN(d_model, d_model, num_layers=num_layers,
        #                     batch_first=True, bidirectional=...)
        self.rnn=nn.RNN(d_model,d_model,num_layers=num_layers,batch_first=True,bidirectional=True)
        # TODO: create nn.Linear(rnn_output_dim, self.output_vocab)
        self.fc=nn.Linear(d_model*2,self.output_vocab)
        return
        raise NotImplementedError('Define embedding, nn.RNN, and output projection')

    def forward(self, x):
        # TODO: x has shape (B, T)
        # TODO: embed -> run nn.RNN -> project all time steps
        emb=self.embedding(x)
        out,_=self.rnn(emb)
        logits=self.fc(out)
        # TODO: return shape (B, T, output_vocab)
        return logits
        raise NotImplementedError('Implement the forward pass with nn.RNN')

In [20]:
def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(dim=-1)
            correct += (preds == y).sum().item()
            total += y.numel()
    return correct / total

def train(K=2, epochs=20, batch_size=256):
    model = RNNModel(K=K).to(device)
    dataset = CustomDataset(K=K)
    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size
    train_set, test_set = random_split(dataset, [train_size, test_size])
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=64)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f'Epoch {epoch + 1}: loss={total_loss / len(train_loader):.4f}, acc={evaluate(model, test_loader, device) * 100:.2f}%')
    return model

# Run after completing RNNModel:
model = train(K=K)

Epoch 1: loss=1.5315, acc=89.16%
Epoch 2: loss=0.6640, acc=91.35%
Epoch 3: loss=0.2237, acc=98.85%
Epoch 4: loss=0.0918, acc=99.26%
Epoch 5: loss=0.0532, acc=99.45%
Epoch 6: loss=0.0371, acc=99.46%
Epoch 7: loss=0.0286, acc=99.52%
Epoch 8: loss=0.0239, acc=99.44%
Epoch 9: loss=0.0205, acc=99.60%
Epoch 10: loss=1.5905, acc=27.41%
Epoch 11: loss=1.2869, acc=84.12%
Epoch 12: loss=0.7515, acc=89.94%
Epoch 13: loss=0.5458, acc=94.63%
Epoch 14: loss=0.4186, acc=93.37%
Epoch 15: loss=0.3363, acc=98.26%
Epoch 16: loss=0.3015, acc=98.32%
Epoch 17: loss=0.1886, acc=99.62%
Epoch 18: loss=0.2272, acc=99.72%
Epoch 19: loss=0.1255, acc=99.81%
Epoch 20: loss=0.2128, acc=99.28%
